# STREME / TOMTOM / FIMO pipeline

This notebook configures, runs and summarizes the motif pipeline. The reusable
implementation is in `streme_pipeline.py`; the same code is used by the Slurm
array jobs.


## Setup


In [ ]:
from pathlib import Path
import pandas as pd

from streme_pipeline import (
    PipelineConfig,
    PipelineParameters,
    STAGES,
    build_status,
    combine_result_tables,
    discover_fastas,
    run_all_stages,
    run_pipeline_for_fasta,
    save_summaries,
)


In [ ]:
config = PipelineConfig.default(
    project=Path("/s/project/ml4rg_students/2026/project15")
)
config.validate()

fastas = discover_fastas(config.fasta_dir)
parameters = PipelineParameters(
    streme_time=1800,
    minw=6,
    maxw=20,
    nmotifs=10,
    fimo_thresh="1e-4",
    fimo_max_stored_scores=100_000,
    fimo_skip_matched_sequence=True,
)

print(f"Found {len(fastas)} FASTA files")
print("FASTA directory:", config.fasta_dir)
print("Result directory:", config.result_dir)
print("JASPAR database:", config.jaspar_fungi)


The pipeline writes a `.pipeline_done.json` next to each successful
result. New results are only reused when their command and input signatures
match. Existing results from the old notebook have no manifest and are accepted
as a legacy cache by default.


## Status


In [ ]:
status = build_status(config, fastas)
display(status.head())
display(status[list(STAGES)].sum().rename("completed"))

incomplete = status.loc[~status[list(STAGES)].all(axis=1)]
print(f"Incomplete datasets: {len(incomplete)}")
display(incomplete.head(20))


## Test one FASTA


In [ ]:
test_fasta = fastas[0]
test_result = run_pipeline_for_fasta(
    config,
    test_fasta,
    parameters=parameters,
    force=False,
    accept_legacy=True,
)
test_result


## Run locally or in an interactive allocation

Stages are processed separately. This keeps independent FIMO-JASPAR work from
waiting behind STREME in the same worker. With `max_workers=None`, the pipeline
uses `SLURM_CPUS_PER_TASK` when available and otherwise at most four workers.
Set `max_workers` explicitly when the allocation or available memory requires it.


In [ ]:
run_report = run_all_stages(
    config,
    fastas,
    parameters=parameters,
    force=False,
    accept_legacy=True,
    max_workers=None,
)

display(run_report.groupby(["stage", "status"]).size())
display(run_report.loc[run_report["status"] == "failed"].head(20))


## Recommended: submit Slurm arrays

Run from the repository root in an environment that provides Python and pandas:

```bash
MAX_CONCURRENT=20 bash slurm/submit_streme_arrays.sh
```

The submission script determines the FASTA count automatically. STREME and
FIMO-JASPAR start independently; TOMTOM and FIMO-STREME depend on the
corresponding STREME array task. Set `PYTHON_BIN` before submission if `python`
does not point to the project environment.


## Refresh status


In [ ]:
status = build_status(config, fastas)
display(status[list(STAGES)].sum().rename("completed"))
display(status.loc[~status[list(STAGES)].all(axis=1)].head(20))


## Save summaries

By default, FIMO is reduced to hit counts per dataset and motif. This avoids
loading or rewriting all FIMO hits, which can be very large. TOMTOM results are
small enough to combine into one table.


In [ ]:
summary_paths = save_summaries(
    config,
    fastas,
    combine_fimo=False,
)
summary_paths


In [ ]:
tomtom_path = summary_paths["tomtom"]
if tomtom_path:
    tomtom_preview = pd.read_csv(tomtom_path, sep="\t", nrows=30)
    display(tomtom_preview.sort_values(["dataset", "q-value"]).head(30))

for key in ("fimo_jaspar_counts", "fimo_streme_counts"):
    path = summary_paths[key]
    if path:
        print(key)
        display(pd.read_csv(path, sep="\t", nrows=20))


### Optional full FIMO tables

Only create these when downstream analysis needs every hit. The function reads
one input file at a time, so memory use stays bounded, but the output can still
be very large.


In [ ]:
# fimo_jaspar_all = combine_result_tables(
#     config,
#     fastas,
#     "fimo_jaspar_tsv",
#     "fimo_jaspar_all.tsv",
# )
# fimo_streme_all = combine_result_tables(
#     config,
#     fastas,
#     "fimo_streme_tsv",
#     "fimo_streme_all.tsv",
# )
